# Praktikum 5: Sentiment Classification with BERT-mini
In these exercises, you will systematically explore how to use BERT-mini in classification tasks. We will explore to main methods: NLI and training a classification head.

In [ ]:
import torch
from torch.utils.data import DataLoader
import numpy as np

from datasets import load_dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## 1 Loading a Sentiment Dataset (IMDB)

For the sentiment analysis task, we will be using a real-world sentiment analysis dataset:  
**IMDB movie reviews**. This dataset is hosted in Huggingface and is offered at `stanfordnlp/imdb`. This dataset contatins a `text` column contating the document or review, and a `label` with values:

- `0` = negative review  
- `1` = positive review

Use the `download_dataset.py` script if you don't have access to the Internet from your jupyter notebook. 

````bash
$ python scripts/download_dataset.py "stanfordnlp/imdb" data/stanfordnlp_imdb
````

In [ ]:
import pandas as pd
from datasets import load_dataset, load_from_disk

#ds = load_dataset("stanfordnlp/imdb")
ds = load_from_disk("../../data/standfordnlp_imdb")

# Convert splits to pandas DataFrames
train_full = ds["train"]
test_full = ds["test"]

print(train_full)
print(test_full)

In [ ]:
from sklearn.model_selection import train_test_split

def create_split(train_size=20_000, test_size=4_000):
    imdb_train = train_full.train_test_split(
        train_size=train_size,
        stratify_by_column="label",
        seed=42
    )["train"]
    
    # Test split — e.g., 4k samples
    imdb_test = test_full.train_test_split(
        train_size=test_size,
        stratify_by_column="label",
        seed=42
    )["train"]
    
    return imdb_train, imdb_test

imdb_train, imdb_test = create_split()

## 2. Zero-Shot Sentiment Classification using NLI
The first method we will use is use a pretrained NLI model to perform classification. As in the seminar we will load a small NLI model, and then we will build some helper functions that will help us with running th experiments.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Local offline loading (recommended)
nli_path = "../../models/prajjwal1_bert-mini-mnli"

nli_tokenizer = AutoTokenizer.from_pretrained(nli_path)
nli_model = AutoModelForSequenceClassification.from_pretrained(nli_path).to(device)
nli_model.eval()

### 2.1 Helper functions for performing classification

In [ ]:
mnli_id2label = {0: "contradiction", 1: "neutral", 2: "entailment"}
ENTAIL_ID = 2 # The value that is associated with entailment in the model
MAX_LEN = 256 # The (input) sequence length we will use (throughout)

def build_hypothesis(template, label):
    """
    Fill a hypothesis template with a given label.
    Example:
        template = "The sentiment is {}."
        label = "positive"
    Returns:
        "The sentiment is positive."
    """
    return template.format(label)


def nli_entailment_score(premise, hypothesis):
    inputs = nli_tokenizer(
        premise,
        hypothesis,
        return_tensors="pt",
        truncation=True,
        max_length=512,   # max seq lenght to be considered
    ).to(device)
    with torch.no_grad():
        logits = nli_model(**inputs).logits
    probs = logits.softmax(-1).squeeze()
    return probs[ENTAIL_ID].item()




def predict_sentiment_nli(review: str,
                          template: str = "The reviewer felt {} after watching the movie.",
                          pos_word: str = "happy",
                          neg_word: str = "unhappy"):
    """Zero-shot sentiment using NLI."""
    h_pos = build_hypothesis(template, pos_word)
    h_neg = build_hypothesis(template, neg_word)

    p_pos = nli_entailment_score(review, h_pos)
    p_neg = nli_entailment_score(review, h_neg)

    if p_pos >= p_neg:
        return 1, p_pos  # positive
    else:
        return 0, p_neg  # negative

predict_sentiment_nli("The movie was aweful")

### 2.2 Helpers for performing evaluating the NLI model
The helper below recieves the dataset, along with a template for the hypothesis and labels for positive and negative sentiment. Returns classification metrics.

In [ ]:
from datasets import Dataset


def evaluate_nli_sentiment(
    ds, 
    template="The reviewer felt {} about the movie.",
    pos_word="happy",
    neg_word="unhappy"
):
    """Evaluate NLI sentiment classifier on a RAW HuggingFace Dataset."""
    all_preds = []
    all_labels = []

    for ex in tqdm(ds, desc="NLI sentiment eval"):
        review = ex["text"]
        label  = ex["label"]

        pred_label, _ = predict_sentiment_nli(review, template, pos_word, neg_word)

        all_preds.append(pred_label)
        all_labels.append(label)

    acc = accuracy_score(all_labels, all_preds)
    prec, rec, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="macro", zero_division=0
    )

    print(f"[NLI] Accuracy: {acc:.3f}")
    print(f"[NLI] Macro F1: {f1:.3f} (P={prec:.3f}, R={rec:.3f})")

    return {"accuracy": acc, "precision": prec, "recall": rec, "f1_macro": f1}

## 3. Fine-tuned models
The second approach we will explore the to train a clasification head on top of the BERT model. We will have two variants: training the full model (classification head + BERT encoder) and training only the classification head (freezeing the encoding parameters).

### 3.1 Preparing the data
We need to prepare the tokenizer of the model we will use (bert-mini), and encode the dataset in the correct format for the fine-tunning. 

Notice that we are truncating the reviews to MAX_LEN, which is the sequence lenght. 

In [ ]:
BERT_MINI_PATH = "../../models/prajjwal1_bert-mini"

tokenizer = AutoTokenizer.from_pretrained(BERT_MINI_PATH)

def encode_dataset(ds: Dataset, max_length = MAX_LEN):
    """Encode a HF Dataset with 'text' and 'label' into BERT inputs."""
    def _encode_batch(batch):
        enc = tokenizer(
            batch["text"],
            truncation=True,
            padding="max_length",
            max_length=max_length,
        )
        enc["labels"] = batch["label"]
        return enc # "input_ids", "attention_mask", "labels"

    ds_enc = ds.map(_encode_batch, batched=True)
    ds_enc.set_format(
        type="torch",
        columns=["input_ids", "attention_mask", "labels"]
    )
    return ds_enc


imdb_train_enc = encode_dataset(imdb_train)
imdb_test_enc  = encode_dataset(imdb_test)

len(imdb_train_enc), len(imdb_test_enc)

### 3.2 Build model for full-tunning
In the full fine-tuning approach, we take a pretrained BERT encoder and attach a classification head on top. During training, all parameters — both the encoder weights and the classification head — are updated using the downstream task data.

HuggingFace’s `AutoModelForSequenceClassification` is designed exactly for this scenario.
It includes:
- a BERT encoder (loaded with pretrained weights), and
- a classification head (randomly initialized, matching num_labels).

When we call `from_pretrained()`, HuggingFace initializes the encoder with the pretrained checkpoint (e.g., bert-mini) and creates a fresh classification head that we will train from scratch. The helper function below builds the model we will use for full fine-tuning.

In [ ]:
def build_full_finetune_model():
    model = AutoModelForSequenceClassification.from_pretrained(
        BERT_MINI_PATH,
        num_labels=2
    )
    return model

### 3.3 Build model with frozen encoder
In this second approach we use BERT as a **fixed feature extractor**. Only the classification head is trained, while all encoder layers are kept frozen. This makes training much faster and requires fewer resources, but typically gives lower accuracy than full fine-tuning.

We freeze the encoder by setting `requires_grad=False` for all its parameters. The classifier head remains trainable.

In [ ]:
def build_frozen_encoder_model():
    model = AutoModelForSequenceClassification.from_pretrained(
        BERT_MINI_PATH,
        num_labels=2
    )

    # Freeze encoder
    if hasattr(model, "bert"):
        encoder = model.bert
    else:
        encoder = model.base_model  # fallback for some architectures

    for param in encoder.parameters():
        param.requires_grad = False

    return model


### 3.4 Training loop
Both models (full fine-tuning and frozen encoder) use the **same training loop**.  
The only difference is the **optimizer**, which determines which parameters are updated.  
Only parameters with `requires_grad=True` will be trained.

In [ ]:
from tqdm.auto import tqdm

def train_classifier(
    model,
    optimizer,
    train_ds_enc,
    num_epochs=1,
    batch_size=16,
):
    """
    Trains parameters that have requires_grad=True.
    Uses whatever optimizer the caller provides.
    Works for both frozen and full models.
    """
    model.to(device)
    model.train()

    train_loader = DataLoader(train_ds_enc, batch_size=batch_size, shuffle=True)

    for epoch in range(num_epochs):
        total_loss = 0.0

        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            batch = {k: v.to(device) for k, v in batch.items()}

            optimizer.zero_grad()
            outputs = model(**batch)
            loss = outputs.loss
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1} — avg loss: {avg_loss:.4f}")

    return model


### 3.5 Evaluating the fine-tune models
Fine-tuned classifiers output logits directly, so we use a dedicated evaluation
helper that computes accuracy and macro-F1. This function works identically for both variants (full fine-tuning and frozen encoder).

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def evaluate_classifier(
    model,
    ds_enc: Dataset,
    batch_size: int = 16,
):
    """Evaluate a classification model on an encoded dataset."""
    model.to(device)
    model.eval()

    loader = DataLoader(ds_enc, batch_size=batch_size)
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating"):
            labels = batch["labels"].numpy()
            batch = {k: v.to(device) for k, v in batch.items()}
            logits = model(**batch).logits
            preds = logits.argmax(dim=-1).cpu().numpy()

            all_preds.extend(preds)
            all_labels.extend(labels)

    acc = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="macro", zero_division=0
    )

    print(f"Accuracy: {acc:.3f}")
    print(f"Macro F1: {f1:.3f} (P={precision:.3f}, R={recall:.3f})")

    return {
        "accuracy": acc,
        "f1_macro": f1,
        "precision_macro": precision,
        "recall_macro": recall,
    }

## 4. Tasks and Exploration

### Task 1. Compare performance of the three models
Compare the NLI classification with the two fine-tuned models. Below is the code for setting up the component and training the model.

#### Questions:
- What model performed better? Why?
- Which fine-tuned model trained faster?
- Which model has faster inference? (evaluation)


In [ ]:
%%time

# We train the full fine-tunning
clf_full = build_full_finetune_model()

optimizer_full = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, clf_full.parameters()),
    lr=2e-5
)

clf_full = train_classifier(clf_full, optimizer_full, imdb_train_enc, num_epochs=6)
metrics_full = evaluate_classifier(clf_full, imdb_test_enc)

In [ ]:
%%time  

# We train the model with frozen encoder weights
clf_frozen = build_frozen_encoder_model()

optimizer_frozen = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, clf_frozen.parameters()),
    lr=1e-3
)
clf_frozen = train_classifier(clf_frozen, optimizer_frozen, imdb_train_enc, num_epochs=5)
metrics_frozen = evaluate_classifier(clf_frozen, imdb_test_enc)


In [ ]:
# NO need to train the NLI, it is already pretrained. We directly evaluate it
metrics_nli = evaluate_nli_sentiment(imdb_test)
metrics_nli

### Task 2. Training epochs
Train the fine-tuned models for 1, 5, 8 epochs. Record the loss and the classification performance.

#### Questions:
- How does performance changes in these three epoch settings?
- How does loss changes?
- Does the models show any sign of overfitting?

### Task 3. Effect of train set size
Explore how the amount of training data affects model performance.  
Fine-tune the model for **5 epochs** using three different training set sizes:

- 1,000 examples  
- 5,000 examples  
- 20,000 examples  

Record the accuracy (and optionally macro F1) for each setting.

#### Questions

- How much data is needed for good performance?
- How does accuracy change as the training set grows?
- Does the frozen-encoder model also benefit from more data?

In [ ]:
train_size = 1000
train_split, _ = create_split(train_size=train_size)
train_enc = encode_dataset(train_split)


clf_frozen = build_frozen_encoder_model()

optimizer_frozen = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, clf_frozen.parameters()),
    lr=1e-3
)
clf_frozen = train_classifier(clf_frozen, optimizer_frozen, train_enc, num_epochs=5)
metrics_frozen = evaluate_classifier(clf_frozen, imdb_test_enc)

clf_full = build_full_finetune_model()

optimizer_full = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, clf_full.parameters()),
    lr=2e-5
)

clf_full = train_classifier(clf_full, optimizer_full, train_enc, num_epochs=6)
metrics_full = evaluate_classifier(clf_full, imdb_test_enc)

print("\nMetrics frozen=====\n")
print(metrics_frozen)

print("\nMetrics full-training=====\n")
print(metrics_full)

### Task 4. Zero-shot Prompt engineering for NLI
Try different prompt and see if you can improve the default prompt (the results from Task 1). Is the classification sensitive to the prompt?

In [ ]:
template = "The sentiment of the review is {}"
pos_word = "positive"
neg_word = "negative"
metrics_nli = evaluate_nli_sentiment(imdb_test,
                                     template=template,
                                     pos_word=pos_word,
                                     neg_word=neg_word)
metrics_nli